# Notebook 01 — Eksplorasi Data (EDA)

**Tujuan:** Memahami struktur dan karakteristik data DTSEN Kabupaten Lahat sebelum preprocessing.

Tahapan:
1. Load data mentah (372 → 371 baris setelah exclude Talang Jawa)
2. Distribusi desa per kecamatan
3. Hitung fitur proporsi kemiskinan
4. Boxplot & heatmap korelasi 8 fitur
5. Top-10 desa kemiskinan tertinggi
6. Distribusi desil per kecamatan (stacked bar)

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')

# Pastikan utils.py ditemukan
ROOT = Path('../').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import load_dtsen, buat_fitur_proporsi, FEATURE_COLS, DATA_DIR, FIGURES_DIR

# Pastikan direktori output ada
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Matplotlib style
plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

print('Library berhasil diimport.')
print(f'ROOT   : {ROOT}')
print(f'DATA   : {DATA_DIR.resolve()}')
print(f'FIGURES: {FIGURES_DIR.resolve()}')

Library berhasil diimport.
ROOT   : analysis
DATA   : analysis\data\raw
FIGURES: analysis\output\figures


In [2]:
# ── Cell 3: Load Data ──────────────────────────────────────────────────────────
FILE_EXCEL = DATA_DIR / '20_JULI_2026-_Rekap_DTSEN_Kabupaten_Lahat.xls'

df = load_dtsen(FILE_EXCEL)

print(f'Shape   : {df.shape}')
print(f'\nTipe data:')
print(df.dtypes)
print(f'\n5 baris pertama:')
df.head()

ℹ  Desa tanpa profil desil valid (di-exclude):
KECAMATAN   KELURAHAN  JUMLAH_KELUARGA
    Lahat Talang Jawa                1

  DTSEN Kabupaten Lahat — Data Loaded
  Baris (desa/kel)  : 371
  Kolom             : 20
  Kecamatan         : 24
  Total KK          : 142,799
  Total Individu    : 450,323

Shape   : (371, 20)

Tipe data:
KECAMATAN          object
KELURAHAN          object
JUMLAH_KELUARGA     int64
JUMLAH_INDIVIDU     int64
DESIL1_KLG          int64
DESIL1_IND          int64
DESIL2_KLG          int64
DESIL2_IND          int64
DESIL3_KLG          int64
DESIL3_IND          int64
DESIL4_KLG          int64
DESIL4_IND          int64
DESIL5_KLG          int64
DESIL5_IND          int64
DESIL6_10_KLG       int64
DESIL6_10_IND       int64
BELUM_PRMK_KLG      int64
BELUM_PRMK_IND      int64
NONAKTIF_KLG        int64
NONAKTIF_IND        int64
dtype: object

5 baris pertama:


,KECAMATAN,KELURAHAN,JUMLAH_KELUARGA,JUMLAH_INDIVIDU,DESIL1_KLG,DESIL1_IND,DESIL2_KLG,DESIL2_IND,DESIL3_KLG,DESIL3_IND,DESIL4_KLG,DESIL4_IND,DESIL5_KLG,DESIL5_IND,DESIL6_10_KLG,DESIL6_10_IND,BELUM_PRMK_KLG,BELUM_PRMK_IND,NONAKTIF_KLG,NONAKTIF_IND
0,Tanjungsakti Pumu,Kembang Ayun,211,683,32,116,19,70,15,43,28,87,28,96,84,262,5,9,11,7
1,Tanjungsakti Pumu,Gunung Meraksa,457,1558,79,300,38,135,29,99,17,48,108,381,172,571,14,24,51,49
2,Tanjungsakti Pumu,Tanjung Alam,456,1561,61,251,41,145,26,90,21,63,82,312,207,667,18,33,38,33
3,Tanjungsakti Pumu,Batu Rancing,222,718,31,113,26,82,21,64,17,49,30,106,94,301,3,3,15,12
4,Tanjungsakti Pumu,Suban,346,1142,55,214,36,132,32,115,23,65,68,231,120,365,12,20,28,24


In [3]:
# ── Cell 4: Distribusi Desa per Kecamatan ─────────────────────────────────────
desa_per_kec = (
    df.groupby('KECAMATAN')['KELURAHAN']
    .count()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(
    desa_per_kec.index,
    desa_per_kec.values,
    color=sns.color_palette('Blues_r', len(desa_per_kec)),
    edgecolor='white',
    linewidth=0.5,
)
for bar, val in zip(bars, desa_per_kec.values):
    ax.text(
        bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2,
        str(val), va='center', ha='left', fontsize=9
    )
ax.set_xlabel('Jumlah Desa/Kelurahan')
ax.set_title('Distribusi Desa/Kelurahan per Kecamatan\nKabupaten Lahat — DTSEN', fontweight='bold')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
outpath = FIGURES_DIR / '01_distribusi_desa_per_kecamatan.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'Gambar disimpan: {outpath}')
print(desa_per_kec.sort_values(ascending=False).to_string())

Gambar disimpan: analysis\output\figures\01_distribusi_desa_per_kecamatan.png
KECAMATAN
Kikim Timur           32
Lahat                 29
Kota Agung            21
Jarai                 20
Merapi Barat          19
Kikim Barat           19
Pajar Bulan           19
Pagar Gunung          19
Tanjung Sakti Pumi    18
Kikim Selatan         18
Mulak Ulu             16
Gumay Talang          15
Tanjungtebat          14
Tanjungsakti Pumu     14
Merapi Timur          14
Pseksu                11
Lahat Selatan         10
Mulak Sebingkai       10
Gumay Ulu             10
Merapi Selatan         9
Kikim Tengah           9
Pulau Pinang           9
Sukamerindu            9
Muarapayang            7


In [4]:
# ── Cell 5: Hitung Fitur Proporsi ─────────────────────────────────────────────
df_feat = buat_fitur_proporsi(df)

print(f'Shape df_feat: {df_feat.shape}')
print(f'\nStatistik deskriptif 8 fitur proporsi (satuan %):'
      )
df_feat[FEATURE_COLS].describe().round(2)

Shape df_feat: (371, 11)

Statistik deskriptif 8 fitur proporsi (satuan %):


,pct_desil1,pct_desil2,pct_desil3,pct_desil4,pct_desil5,pct_desil6_10,pct_belum_prmk,pct_nonaktif
count,371.00,371.00,371.00,371.00,371.00,371.00,371.00,371.00
mean,15.13,13.53,8.96,7.20,13.27,38.31,3.61,7.97
std,6.68,5.24,3.32,3.00,4.32,11.99,1.66,2.40
min,2.94,2.75,1.69,1.59,2.58,12.84,0.00,2.00
25%,9.97,9.71,6.54,5.06,10.38,29.66,2.40,6.45
50%,14.94,13.38,8.79,6.69,12.84,36.77,3.45,7.78
75%,19.37,16.78,11.03,8.96,15.55,46.41,4.64,9.44
max,44.37,35.78,21.65,17.54,29.41,74.52,12.83,16.92


In [5]:
# ── Cell 6: Boxplot 8 Fitur Proporsi ──────────────────────────────────────────
label_map = {
    'pct_desil1':     'Desil 1 (%)',
    'pct_desil2':     'Desil 2 (%)',
    'pct_desil3':     'Desil 3 (%)',
    'pct_desil4':     'Desil 4 (%)',
    'pct_desil5':     'Desil 5 (%)',
    'pct_desil6_10':  'Desil 6–10 (%)',
    'pct_belum_prmk': 'Belum PRMK (%)',
    'pct_nonaktif':   'Nonaktif KLG (%)',
}

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    ax = axes[i]
    data = df_feat[col].dropna()
    bp = ax.boxplot(
        data,
        patch_artist=True,
        boxprops=dict(facecolor='#4C9BE8', alpha=0.7),
        medianprops=dict(color='#E84C4C', linewidth=2),
        whiskerprops=dict(color='#555'),
        capprops=dict(color='#555'),
        flierprops=dict(marker='o', markerfacecolor='gray', markersize=3, alpha=0.5),
    )
    ax.set_title(label_map[col], fontsize=9)
    ax.set_xticks([])
    ax.set_ylabel('%', fontsize=8)
    median_val = data.median()
    ax.axhline(median_val, color='#E84C4C', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.text(1.35, median_val, f'{median_val:.1f}', va='center', fontsize=7, color='#E84C4C')

fig.suptitle('Distribusi 8 Fitur Proporsi Kemiskinan (Boxplot)', fontweight='bold', y=1.01)
plt.tight_layout()
outpath = FIGURES_DIR / '01_boxplot_fitur.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'Gambar disimpan: {outpath}')

Gambar disimpan: analysis\output\figures\01_boxplot_fitur.png


In [6]:
# ── Cell 7: Heatmap Korelasi 8 Fitur ──────────────────────────────────────────
corr = df_feat[FEATURE_COLS].corr()

label_short = [
    'Desil1', 'Desil2', 'Desil3', 'Desil4',
    'Desil5', 'Desil6-10', 'BlmPRMK', 'Nonaktif'
]

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr,
    ax=ax,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    linewidths=0.5,
    xticklabels=label_short,
    yticklabels=label_short,
    square=True,
    cbar_kws={'shrink': 0.8},
)
ax.set_title('Heatmap Korelasi Pearson — 8 Fitur Proporsi', fontweight='bold')
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
outpath = FIGURES_DIR / '01_heatmap_korelasi.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'Gambar disimpan: {outpath}')
print('\nMatriks Korelasi:')
corr.round(3)

Gambar disimpan: analysis\output\figures\01_heatmap_korelasi.png

Matriks Korelasi:


,pct_desil1,pct_desil2,pct_desil3,pct_desil4,pct_desil5,pct_desil6_10,pct_belum_prmk,pct_nonaktif
pct_desil1,1.000,0.444,0.217,0.044,-0.193,-0.737,-0.114,0.188
pct_desil2,0.444,1.000,0.519,0.353,-0.276,-0.777,-0.291,0.038
pct_desil3,0.217,0.519,1.000,0.539,-0.285,-0.622,-0.248,-0.006
pct_desil4,0.044,0.353,0.539,1.000,-0.306,-0.445,-0.167,0.036
pct_desil5,-0.193,-0.276,-0.285,-0.306,1.000,0.027,-0.028,-0.124
pct_desil6_10,-0.737,-0.777,-0.622,-0.445,0.027,1.000,0.173,-0.126
pct_belum_prmk,-0.114,-0.291,-0.248,-0.167,-0.028,0.173,1.000,0.305
pct_nonaktif,0.188,0.038,-0.006,0.036,-0.124,-0.126,0.305,1.000


In [7]:
# ── Cell 8: Top-10 Desa Kemiskinan Tertinggi ───────────────────────────────────
df_feat['pct_miskin'] = df_feat['pct_desil1'] + df_feat['pct_desil2']

top10 = (
    df_feat[['KECAMATAN', 'KELURAHAN', 'pct_desil1', 'pct_desil2', 'pct_miskin']]
    .sort_values('pct_miskin', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top10.index = top10.index + 1
print('Top-10 Desa/Kelurahan dengan Kemiskinan Tertinggi (pct_desil1 + pct_desil2):')
print(top10.to_string())

# Bar chart horizontal
fig, ax = plt.subplots(figsize=(10, 5))
labels = [f"{row.KELURAHAN}\n({row.KECAMATAN})" for _, row in top10.iterrows()]
y_pos  = range(len(top10))

bars1 = ax.barh(y_pos, top10['pct_desil1'], label='Desil 1', color='#D62728', alpha=0.85)
bars2 = ax.barh(y_pos, top10['pct_desil2'], left=top10['pct_desil1'],
                label='Desil 2', color='#FF7F0E', alpha=0.85)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Proporsi KK (%)')
ax.set_title('Top-10 Desa/Kelurahan Kemiskinan Tertinggi\n(Desil 1 + Desil 2)', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)

for i, (_, row) in enumerate(top10.iterrows()):
    ax.text(row['pct_miskin'] + 0.3, i, f"{row['pct_miskin']:.1f}%", va='center', fontsize=8)

plt.tight_layout()
outpath = FIGURES_DIR / '01_top10_desa_termiskin.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'\nGambar disimpan: {outpath}')

Top-10 Desa/Kelurahan dengan Kemiskinan Tertinggi (pct_desil1 + pct_desil2):
             KECAMATAN           KELURAHAN  pct_desil1  pct_desil2  pct_miskin
1          Muarapayang         Bandu Agung   39.436620   18.779343   58.215962
2        Kikim Selatan    Beringin Janggut   44.366197   13.380282   57.746479
3       Merapi Selatan             Geramat   17.737003   35.779817   53.516820
4   Tanjung Sakti Pumi         Negeri Kaya   26.000000   26.666667   52.666667
5         Tanjungtebat   Pandan Arang Ilir   29.378531   20.903955   50.282486
6         Tanjungtebat       Padang Perigi   30.290456   19.917012   50.207469
7               Pseksu         Lubuk Atung   32.972973   16.216216   49.189189
8            Mulak Ulu       Tebing Tinggi   22.807018   26.315789   49.122807
9           Kota Agung    Tanjung Beringin   27.205882   21.323529   48.529412
10           Mulak Ulu  Lawang Agung Mulak   29.403794   19.105691   48.509485



Gambar disimpan: analysis\output\figures\01_top10_desa_termiskin.png


In [8]:
# ── Cell 9: Stacked Bar — Distribusi Desil per Kecamatan ──────────────────────
DESIL_COLS = ['pct_desil1','pct_desil2','pct_desil3','pct_desil4',
              'pct_desil5','pct_desil6_10','pct_belum_prmk']
DESIL_LABELS = ['Desil 1','Desil 2','Desil 3','Desil 4',
                'Desil 5','Desil 6-10','Belum PRMK']
DESIL_COLORS = ['#D62728','#FF7F0E','#FFBB78','#2CA02C',
                '#98DF8A','#1F77B4','#AEC7E8']

agg = (
    df_feat.groupby('KECAMATAN')[DESIL_COLS]
    .mean()
    .sort_values('pct_desil1', ascending=False)
)

fig, ax = plt.subplots(figsize=(12, 7))
bottom = np.zeros(len(agg))
for col, label, color in zip(DESIL_COLS, DESIL_LABELS, DESIL_COLORS):
    vals = agg[col].values
    ax.barh(agg.index, vals, left=bottom, label=label, color=color, alpha=0.88, edgecolor='white', linewidth=0.3)
    bottom += vals

ax.invert_yaxis()
ax.set_xlabel('Rata-rata Proporsi KK (%)')
ax.set_title(
    'Distribusi Rata-rata Desil per Kecamatan (Stacked Bar)\n'
    'Kabupaten Lahat — DTSEN',
    fontweight='bold'
)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, title='Kategori')
ax.set_yticklabels(agg.index, fontsize=8)
plt.tight_layout()
outpath = FIGURES_DIR / '01_distribusi_desil_kecamatan.png'
plt.savefig(outpath, dpi=150, bbox_inches='tight')
plt.close()
print(f'Gambar disimpan: {outpath}')

Gambar disimpan: analysis\output\figures\01_distribusi_desil_kecamatan.png


## Ringkasan Temuan EDA

- **Jumlah data:** 371 desa/kelurahan di 24 kecamatan (1 desa Talang Jawa di-exclude karena KK=0)
- **Distribusi kecamatan:** Bervariasi; beberapa kecamatan memiliki >20 desa, beberapa hanya <10
- **Fitur proporsi:** Terdapat variasi yang signifikan pada pct_desil1 dan pct_desil2 antar desa
- **Korelasi:** pct_desil1 dan pct_desil2 cenderung berkorelasi positif (desa miskin di desil 1 juga tinggi di desil 2); pct_desil6_10 berkorelasi negatif dengan desil rendah
- **Top-10 desa termiskin:** Beberapa desa memiliki pct_desil1+2 sangat tinggi (>40%), menunjukkan konsentrasi kemiskinan ekstrem
- **Kecamatan:** Kecamatan tertentu mendominasi desa-desa dengan proporsi Desil 1 tinggi → perlu prioritas intervensi

> **Langkah selanjutnya:** Preprocessing (NB02) — normalisasi fitur untuk clustering